# Export the locked retrieval artifacts (Kaggle)

Builds, once, everything the app needs to serve the configuration the Phase 1
tournament locked in, and packages it as a single ZIP to download.

| exported | what it is | built by |
| --- | --- | --- |
| `chunks.jsonl` | the chunked article text, recursive 512 / 64 | `materialize_evaluation_dataset.py` |
| `bge_m3_sparse.pkl` | BGE-M3 learned-sparse vectors over those chunks | `build_retrieval_models_index.py` |
| `chroma_db/` | dense vector database for the fallback route | `build_retrieval_models_index.py` |
| `testset_resolved.jsonl` | the resolved question set, for local re-scoring | `materialize_evaluation_dataset.py` |
| `bundle_manifest.json` | SHA-256 of every file above, plus provenance | this notebook |

The locked configuration comes from the repository's own `configs/config.yaml`
at the pinned commit — it is not restated here, so the export cannot drift from
what the app reads.

**Runtime:** GPU T4 x1. Expect roughly 25–45 minutes; the BGE-M3 encode of
every chunk dominates. Internet must be **on** (the dataset and model weights
are downloaded). It resumes: re-running skips any stage whose output exists.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='faac9d64a0eafaa21e66a3726d49757f4425a91c'  # the refactor; earlier commits point pyproject at a backend/ that is gone
HF_REPO_ID='MatchaMacchiato/newsqa_200_11064_v2.0.0'
HF_REVISION='b81c8db6847a23272665946c0c43c72e9a212fd9'  # the v2.0.0 commit; swap for 'v2.0.0' once that tag exists

SPARSE_ID='bge_m3_sparse'          # matches the filename the app looks for
BUILD_DENSE_CHROMA=True            # the fallback route; set False to export sparse only
ARTIFACT_VERSION='locked-bge-m3-512-64-v2'

# Recorded on the first run and pasted back here to lock every later rebuild
# to a byte-exact result. They belong to ONE dataset revision: restoring
# article text changes the chunk count and therefore both files below.
EXPECTED_CHUNKS=None
EXPECTED_CHUNKS_SHA256=None

FORCE_REBUILD=False
KAGGLE_WORKING=Path('/kaggle/working')
PROJECT_ROOT=KAGGLE_WORKING/'Text-Mining---NewsQA-RAG'
WORK_ROOT=KAGGLE_WORKING/'newsqa_locked_export'
VARIANT_ROOT=WORK_ROOT/'data'
SPARSE_DIR=WORK_ROOT/'index_sparse'   # separate dirs: the builder overwrites
DENSE_DIR=WORK_ROOT/'index_dense'     # index_manifest.json on every run
EXPORT_ROOT=WORK_ROOT/'export'
LOG_ROOT=WORK_ROOT/'logs'
BUNDLE=KAGGLE_WORKING/f'{ARTIFACT_VERSION}.zip'
MIN_FREE_GIB=5

## 1. Runtime setup

Turn on **GPU T4 x1** and **Internet** in the notebook settings. The dataset is
public, so no token is needed; set the Kaggle secret `HF_TOKEN` only to lift
anonymous download rate limits.

In [ ]:
# The dataset is public, so a token is optional. Set one as the Kaggle
# secret HF_TOKEN only to lift anonymous download rate limits.
token=''
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception:
    pass
if token:
    os.environ['HF_TOKEN']=token
else:
    print('No HF_TOKEN secret; downloading the public dataset anonymously.')
os.environ['HF_HOME']=str(KAGGLE_WORKING/'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1'})

if FORCE_REBUILD and WORK_ROOT.exists(): shutil.rmtree(WORK_ROOT)
for path in [VARIANT_ROOT,SPARSE_DIR,DENSE_DIR,EXPORT_ROOT,LOG_ROOT]: path.mkdir(parents=True,exist_ok=True)

if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=300)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)

import torch, yaml
assert torch.cuda.is_available(), 'Enable a T4 GPU before building the BGE-M3 index'
print('GPU:',torch.cuda.get_device_name(0),round(torch.cuda.get_device_properties(0).total_memory/2**30,1),'GiB')
print('Pinned commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())

In [ ]:
LOCKED_CONFIG=PROJECT_ROOT/'configs/config.yaml'
config=yaml.safe_load(LOCKED_CONFIG.read_text(encoding='utf-8'))
CHUNK=config['chunking']
RETRIEVAL=config['retrieval']
DENSE_MODEL=config['embedding']['model_name']

# Read the lock, do not restate it. If the repository config ever stops being
# the tournament winner, this fails here rather than exporting the wrong index.
assert RETRIEVAL['retriever']=='sparse', RETRIEVAL['retriever']
assert RETRIEVAL['sparse']['method']=='bge-m3', RETRIEVAL['sparse']
assert (CHUNK['strategy'],CHUNK['chunk_size'],CHUNK['chunk_overlap'])==('recursive',512,64), CHUNK
print('chunking :',CHUNK['strategy'],CHUNK['chunk_size'],'/',CHUNK['chunk_overlap'])
print('retriever:',RETRIEVAL['retriever'],'| method',RETRIEVAL['sparse']['method'],'| top_k',RETRIEVAL['top_k'])
print('reranker :',RETRIEVAL['reranker']['model'],'-> top_n',RETRIEVAL['reranker']['top_n'])
print('dense    :',DENSE_MODEL,'(fallback route)' if BUILD_DENSE_CHROMA else '(skipped)')

In [ ]:
def sha256_file(path,block_size=1<<20):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()

def jsonl_count(path):
    with Path(path).open(encoding='utf-8') as handle: return sum(1 for line in handle if line.strip())

def disk_status(label=''):
    usage=shutil.disk_usage(KAGGLE_WORKING); free=usage.free/2**30
    print(f'Disk {label}: {free:.1f} GiB free of {usage.total/2**30:.1f}',flush=True)
    assert free>MIN_FREE_GIB, f'Only {free:.1f} GiB left; free space before continuing'
    return free

def run_command(command,label):
    command=[str(value) for value in command]
    log_path=LOG_ROOT/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('$',' '.join(command),flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=os.environ.copy(),stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); log.flush()
        returncode=process.wait()
    if returncode: raise subprocess.CalledProcessError(returncode,command)
    return log_path

def lock(label,observed,expected):
    if expected is None:
        print(f'  RECORD  {label} = {observed!r}'); return observed
    assert observed==expected, f'{label} drifted: {observed!r} != {expected!r}'
    print(f'  locked  {label} ok'); return observed

disk_status('start')

## 2. Materialize and chunk the corpus

Downloads the pinned public v2.0.0 dataset and chunks all 11,064 articles at
512 / 64. Question deduplication is skipped: it exists to keep near-duplicate
questions out of an evaluation average, the app never reads it, and its
human-approval record is sealed against one corpus revision. Chunks are
byte-identical either way — the deduplicator copies `chunks.jsonl` through
untouched.

The baseline Chroma build is skipped here too; section 4 builds it directly
from the finished chunks instead of re-chunking to get there.

In [ ]:
CHUNKS=VARIANT_ROOT/'final/chunks.jsonl'
RESOLVED=VARIANT_ROOT/'final/testset_resolved.jsonl'
VARIANT_MANIFEST=VARIANT_ROOT/'manifests/variant.json'

if not CHUNKS.exists():
    run_command([sys.executable,'-u','scripts/materialize_evaluation_dataset.py',
                 '--repo-id',HF_REPO_ID,'--revision',HF_REVISION,
                 '--config',LOCKED_CONFIG,'--output-root',VARIANT_ROOT,
                 '--db-path',WORK_ROOT/'temporary_chroma',
                 '--skip-vector-index','--no-deduplicate'],'materialize')
else:
    print('Chunks already exist; skipping materialization.')

OBSERVED_CHUNKS=lock('EXPECTED_CHUNKS',jsonl_count(CHUNKS),EXPECTED_CHUNKS)
OBSERVED_CHUNKS_SHA256=lock('EXPECTED_CHUNKS_SHA256',sha256_file(CHUNKS),EXPECTED_CHUNKS_SHA256)
if EXPECTED_CHUNKS is None:
    print('\nPaste these into the configuration cell to lock this revision:')
    print('EXPECTED_CHUNKS='+str(OBSERVED_CHUNKS))
    print('EXPECTED_CHUNKS_SHA256='+repr(OBSERVED_CHUNKS_SHA256))
print('\nresolved questions:',jsonl_count(RESOLVED))
disk_status('after chunking')

## 3. Build the BGE-M3 learned-sparse index

The expensive step, and the one the locked configuration actually retrieves
with. A T4 encodes every chunk; only the lexical postings are persisted, not
dense vectors. The builder can sit quiet for minutes while the GPU works —
watch Kaggle's resource panel rather than interrupting it.

In [ ]:
SPARSE_INDEX=SPARSE_DIR/f'{SPARSE_ID}.pkl'
if not SPARSE_INDEX.exists():
    run_command([sys.executable,'-u','scripts/build_retrieval_models_index.py',
                 '--chunks-path',CHUNKS,'--base-config',LOCKED_CONFIG,
                 '--base-variant-manifest',VARIANT_MANIFEST,
                 '--output-dir',SPARSE_DIR,'--sparse-ids',SPARSE_ID,
                 '--skip-dense','--device','cuda'],'build_sparse')
else:
    print('BGE-M3 index already exists; skipping encoding.')
print('sparse index:',round(SPARSE_INDEX.stat().st_size/2**20,1),'MiB')
disk_status('after sparse')

## 4. Build the dense Chroma collection

The vector database behind the app's fallback `dense` route. Round 1 measured
this embedder as the weakest of the four tested, so it is a fallback and not
the recommendation — it is exported because the app still offers the route and
the collection has to match the same chunks.

In [ ]:
DENSE_SLUG=DENSE_MODEL.replace('/','_').replace('-','_').lower()
CHROMA_DIR=DENSE_DIR/f'chroma_{DENSE_SLUG}'
COLLECTION_NAME=f'chunks_{DENSE_SLUG}'
if BUILD_DENSE_CHROMA and not CHROMA_DIR.exists():
    run_command([sys.executable,'-u','scripts/build_retrieval_models_index.py',
                 '--chunks-path',CHUNKS,'--base-config',LOCKED_CONFIG,
                 '--base-variant-manifest',VARIANT_MANIFEST,
                 '--output-dir',DENSE_DIR,'--dense-models',DENSE_MODEL,
                 '--skip-sparse','--device','cuda'],'build_dense')
elif BUILD_DENSE_CHROMA:
    print('Chroma collection already exists; skipping encoding.')
else:
    print('BUILD_DENSE_CHROMA is False; no vector database was built.')
print('collection:',COLLECTION_NAME,'at',CHROMA_DIR,'| exists:',CHROMA_DIR.exists())
disk_status('after dense')

## 5. Validate, then assemble the bundle

Load the sparse index back and check it covers every chunk before packaging.
An index that silently indexed a subset is the failure worth catching here,
not after the download.

In [ ]:
sys.path.insert(0,str(PROJECT_ROOT/'common'))
from newsqa_rag.indexing.learned_sparse_index import LearnedSparseIndex

index_manifest=json.loads((SPARSE_DIR/'index_manifest.json').read_text(encoding='utf-8'))
assert index_manifest['total_chunks']==OBSERVED_CHUNKS, index_manifest['total_chunks']
assert index_manifest['chunks_sha256']==OBSERVED_CHUNKS_SHA256
record=index_manifest['sparse_indexes'][SPARSE_ID]
assert record['method']=='bge-m3' and record['model_name']==RETRIEVAL['sparse']['model'], record

loaded=LearnedSparseIndex.load(str(SPARSE_INDEX),device='cuda')
assert loaded.size==OBSERVED_CHUNKS, f'index holds {loaded.size} of {OBSERVED_CHUNKS} chunks'
print(f'sparse index verified: {loaded.size} chunks')

# One end-to-end query, so a broken artifact fails here and not in the app.
probe=loaded.query('who was elected president', 5)
assert len(probe)==5 and probe[0]['score']>0, probe
print('probe query returned',len(probe),'hits, top score',round(probe[0]['score'],4))
del loaded
disk_status('after validation')

In [ ]:
# The app resolves artifacts by filename, so name them what it looks for
# rather than making it configurable at both ends.
shutil.copy2(CHUNKS,EXPORT_ROOT/'chunks.jsonl')
shutil.copy2(SPARSE_INDEX,EXPORT_ROOT/'bge_m3_sparse.pkl')
shutil.copy2(RESOLVED,EXPORT_ROOT/'testset_resolved.jsonl')
if BUILD_DENSE_CHROMA and CHROMA_DIR.exists():
    target=EXPORT_ROOT/'chroma_db'
    if target.exists(): shutil.rmtree(target)
    shutil.copytree(CHROMA_DIR,target)

files=sorted(path for path in EXPORT_ROOT.rglob('*') if path.is_file())
manifest={
    'schema_version':1,
    'artifact_version':ARTIFACT_VERSION,
    'created_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
    'source':{'hf_repo_id':HF_REPO_ID,'hf_revision':HF_REVISION,'repo_commit':REPO_COMMIT},
    'pipeline':{'chunking':CHUNK,'retrieval':RETRIEVAL,'dense_model':DENSE_MODEL if BUILD_DENSE_CHROMA else None},
    'chroma':{'collection':COLLECTION_NAME,'path':'chroma_db'} if BUILD_DENSE_CHROMA else None,
    'statistics':{'chunks':OBSERVED_CHUNKS,'resolved_questions':jsonl_count(RESOLVED)},
    'artifacts':{str(path.relative_to(EXPORT_ROOT)).replace('\\','/'):
                 {'bytes':path.stat().st_size,'sha256':sha256_file(path)} for path in files},
}
(EXPORT_ROOT/'bundle_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print(json.dumps({k:v for k,v in manifest.items() if k!='artifacts'},indent=2))
print('\nfiles:',len(files)+1)

## 6. Package and download

Writes one ZIP to `/kaggle/working`. Download it from the notebook's **Output**
panel — or, on a committed run, from the version's output files.

In [ ]:
temporary=BUNDLE.with_suffix('.zip.tmp')
with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
    for path in sorted(EXPORT_ROOT.rglob('*')):
        if path.is_file(): archive.write(path,path.relative_to(EXPORT_ROOT))
temporary.replace(BUNDLE)

print('bundle   :',BUNDLE)
print('size     :',round(BUNDLE.stat().st_size/2**20,1),'MiB')
print('sha256   :',sha256_file(BUNDLE))
disk_status('final')

## Installing it into the app

Unpack the ZIP into `data/locked_index/` at the repository root:

```bash
mkdir -p data/locked_index
unzip locked-bge-m3-512-64-v2.zip -d data/locked_index
```

`configs/config.yaml` already points at that directory (`artifacts.locked_index_dir`),
so the API picks it up on the next start — `/retrieval/algorithms` flips
`locked` to available, and chat switches from the Chroma route to the locked
one. Override the location with `RAG_LOCKED_INDEX_DIR` if you keep it elsewhere.

To use the exported Chroma collection for the fallback `dense` route as well:

```bash
RAG_DB_PATH=data/locked_index/chroma_db
RAG_COLLECTION=chunks_all_minilm_l6_v2   # the exact name is in bundle_manifest.json
```

First query is slow — BGE-M3 and `bge-reranker-large` both load weights on
demand, and on CPU the reranker is the slow half. Set
`retrieval.reranker.device` in the config if the host has a GPU.